In [ ]:
# Ensure PyTorch is available in this notebook kernel
import importlib.util
import subprocess
import sys

from pathlib import Path

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])

from torch.utils.data import ConcatDataset
from myAPI.mnist_to_pt import pt_mnist

import torch
print("torch:", torch.__version__)
from torch.utils.data import ConcatDataset

torch: 2.11.0+cpu


In [3]:
from myAPI.model import MNISTDataset, MyData, CNN, device

import torch

import json

# device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [4]:
# Load model config
from pathlib import Path

configFile = Path("sampleModel") / "config.json"
with open(configFile, "r") as f:
    config = json.load(f)


In [ ]:
# create a model
model = CNN(**config["model"]).to(device)

# create mnist dataset path is the folder that contain all four files train = false to get the test data
mcao_data = MNISTDataset(Path("mydata") / "Minh Cao", train=True)
mngo_data = MNISTDataset(Path("mydata") / "Minh Ngo", train=True)
lkiet_data = MNISTDataset(Path("mydata") / "Le Kiet", train=True)

mcao_data_test = MNISTDataset(Path("mydata") / "Minh Cao", train=False)
mngo_data_test = MNISTDataset(Path("mydata") / "Minh Ngo", train=False)
lkiet_data_test = MNISTDataset(Path("mydata") / "Le Kiet", train=False)

lkhoi_data_test = MNISTDataset(Path("mydata") / "Le Khoi", train=False)
ghuy_data_test = MNISTDataset(Path("mydata") / "Gia Huy", train=False)

trainData = ConcatDataset([mcao_data, mngo_data, lkiet_data])
trainDataMnist = MNISTDataset("mnist", train=True)

testData1 = ConcatDataset([mcao_data_test, mngo_data_test, lkiet_data_test])
testData2 = lkhoi_data_test
testData3 = ghuy_data_test
testDataMnist = MNISTDataset("mnist", train=False)

In [ ]:
# idx = 10
# trainDataMnist[idx][1]

pt_mnist(trainDataMnist, "tests/new-ubyte")

In [50]:
# use the training_process method to train, pass in the dataset and the training config
# training process returns the loss history and acc_history
[loss_hist, acc_hist] = model.training_process(trainDataMnist,**config["training"])

Training progress: 100%|██████████| 5/5 [00:47<00:00,  9.50s/it]


In [51]:
model.save_weight("newWeigh.pth")

In [52]:
print(loss_hist)

[0.23532704512576264, 0.06179248773381114, 0.04359927739041547, 0.034225842201182, 0.027325236348140363]


In [60]:
# testing_process takes in the data and returns loss, accuracy, conf_matrix,
# wrong indices, and (optionally) wrong_details with true/pred labels

# testMndata = MNISTDataset(path=data_path, train=False)
testData = testData3
[loss, acc, conf_mat, wrong_idx, wrong_details] = model.testing_process(
    testData,
    return_wrong_details=True,
)

In [61]:
print(acc)

0.951048951048951


In [47]:
# --- Export results (train/test loss, confusion matrix, wrong samples) ---
from myAPI.results_export import export_results

export_dir = export_results(
    out_dir="exports",
    train_loss_history=loss_hist,
    train_accuracy_history=acc_hist,
    test_loss=loss,
    test_accuracy=acc,
    confusion_matrix=conf_mat,
    wrong_indices=wrong_idx,
    wrong_details=wrong_details,
    train_dataset=trainData,
    test_dataset=testData,
    extra={"config": config, "device": device},
)
print("Exported to:", export_dir)


Exported to: exports\run_20260519_001952


In [ ]:
from pathlib import Path

weightFile = Path("sampleModel") / "model_weights.pth"
model.load_weight(weightFile)
model.clear_weight()
model.save_weight(Path("newWeigh.pth"))
